In [1]:
import os
import sys

# 1. Paths
project_root = r"C:\Users\jamba\Documents\Mlops\Mlops-end"
src_path = r"C:\Users\jamba\Documents\Mlops\Mlops-end\src"

# 2. Change the working directory so YAML config files are found
if os.getcwd() != project_root:
    os.chdir(project_root)

# 3. Add the 'src' folder to Python's path so imports are found
if src_path not in sys.path:
    sys.path.insert(0, src_path)


In [2]:
%pwd

'C:\\Users\\jamba\\Documents\\Mlops\\Mlops-end'

In [3]:
from src.mlopsProject.constants import *
from src.mlopsProject.utils.common import read_yaml, create_directories
from src.mlopsProject import logger


In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataConfig:
    root_dir: Path
    source_URL : str
    local_data_file : Path
    unzip_dir: Path

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [6]:
import os
import urllib.request as request
import zipfile
from mlopsProject import logger
from mlopsProject.utils.common import get_size

In [7]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
  

In [8]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-04-14 23:09:14,116: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-14 23:09:14,121: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-14 23:09:14,124: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-14 23:09:14,127: INFO: common: created directory at: artifacts]
[2026-04-14 23:09:14,130: INFO: common: created directory at: artifacts/data_ingestion]


NameError: name 'DataIngestionConfig' is not defined